# 04 - Model Experiments
Compare candidate models with cross-validation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score
from src.config import load_config
from src.features.build_features import add_engineered_features
from src.models.train_model import build_pipeline

In [ ]:
cfg = load_config()
df = add_engineered_features(pd.read_csv('../data/processed/churn_clean.csv'))
X, y = df.drop(columns=['churn']), df['churn']

In [ ]:
results = {}
for name, params in {
    'random_forest': {'n_estimators': 200, 'max_depth': 10, 'class_weight': 'balanced', 'random_state': 42},
    'gradient_boosting': {'n_estimators': 150, 'learning_rate': 0.1, 'random_state': 42},
    'logistic_regression': {'class_weight': 'balanced'},
}.items():
    cfg['model'] = {'type': name, 'params': params}
    results[name] = cross_val_score(build_pipeline(cfg), X, y, cv=5, scoring='roc_auc').mean()
pd.Series(results).sort_values().plot.barh(title='CV ROC-AUC')
plt.show()